In [2]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    # base_url="",
)

### What is RAG - Why LLMs need context, the RAG architecture

In [5]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [6]:
llm("Hey, what's up?")

'Not much—just here and ready to help. What’s going on?'

In [7]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Possibly — it depends on the course’s enrollment rules and whether the class is still open.

If you want, I can help you figure it out. Usually the key questions are:
- Is the course still accepting new students?
- Has the registration deadline passed?
- Is there a waitlist or late-add option?
- Do you need instructor permission?

If you’d like, send me the course name or the platform/school, and I can help you draft a quick message asking whether you can still join.


In [8]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [9]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

### The Course FAQ Dataset - Fetching and exploring the FAQ data

In [10]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [14]:
courses_raw[:3]

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41}]

In [16]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [20]:
documents[:2]

[{'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: When does the course start?',
  'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."},
 {'id': 'bfafa427b3',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: What are the prerequisites for this course?',
  'answer': "To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful 

### Search basics

In [22]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [24]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [25]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [26]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [27]:
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

In [29]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [32]:
search_results = search("How do I submit the homeworks?")
print(search_results)

[{'id': 'd5fc98925d', 'course': 'llm-zoomcamp', 'section': 'Capstone Project', 'question': 'Do we submit 2 projects, what does attempt 1 and 2 mean?', 'answer': 'You only need to submit one project. If the submission at the first attempt fails, you can improve it and re-submit during the attempt#2 submission window.\n\n- If you want to submit two projects for the experience and exposure, you must use different datasets and problem statements.\n- If you can’t make it to the attempt#1 submission window, you still have time to catch up to meet the attempt#2 submission window.\n\nRemember that the submission does not count towards the certification if you do not participate in the peer-review of three peers in your cohort.'}, {'id': 'e394e6f738', 'course': 'llm-zoomcamp', 'section': 'Workshop: Open-Source Data Ingestion (dlt)', 'question': 'How do I know which tables are in the db?', 'answer': 'You can use the `db.table_names()` method to list all the tables in the database.'}, {'id': '049

### Building the Prompt

In [33]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [34]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [35]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [38]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [39]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
Capstone Project
Q: Do we submit 2 projects, what does attempt 1 and 2 mean?
A: You only need to submit one project. If the submission at the first attempt fails, you can improve it and re-submit during the attempt#2 submission window.

- If you want to submit two projects for the experience and exposure, you must use different datasets and problem statements.
- If you can’t make it to the attempt#1 submission window, you still have time to catch up to meet the attempt#2 submission window.

Remember that the submission does not count towards the certification if you do not participate in the peer-review of three peers in your cohort.

Workshop: Open-Source Data Ingestion (dlt)
Q: How do I know which tables are in the db?
A: You can use the `db.table_names()` method to list all the tables in the database.

General Course-Related Questions
Q: How should I start the course and follow the weekly workflow?
A: Start with the [

### The LLM

In [40]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [52]:
response.output[0].content[0].text

'Yes — you can join now.\n\nThe course materials are available anytime, and the usual advice is to start with the docs, videos, and GitHub repo, then follow the weekly workflow at your own pace. If you’ve joined late, you can still catch up using the course platform deadlines and materials.\n\nIf you want, I can also help you figure out the best way to catch up quickly.'

In [53]:
response.output_text

'Yes — you can join now.\n\nThe course materials are available anytime, and the usual advice is to start with the docs, videos, and GitHub repo, then follow the weekly workflow at your own pace. If you’ve joined late, you can still catch up using the course platform deadlines and materials.\n\nIf you want, I can also help you figure out the best way to catch up quickly.'

In [54]:
response.usage

ResponseUsage(input_tokens=915, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=84, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=999)

In [55]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00106425

In [57]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

In [59]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

In [60]:
response.output_text

'Yes — you can start whenever you want. The videos, GitHub materials, and deadlines are available in the course management platform.'

In [61]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
    
    response = openai_client.responses.create(
        model=model,
        input=message_history
    )
    return response.output_text

In [62]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model)
    return answer

In [63]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join and start learning now.

If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [64]:
print(rag("Where can I ask questions with other learners taking the course?"))

You can ask technical questions with other learners in the course **Slack channel**.


In [65]:
print(rag("How do I get a certificate?"))

You can get a certificate only if you finish the course with a live cohort and pass the Capstone project.

Self-paced mode does not include a certificate, because certificates require peer-reviewing 3 capstones during the live course period.


### RAG Helper

In [66]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.


In [67]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

In [68]:
assistant.rag("How do I get a certificate?")

'To get a certificate, you need to finish the course with a **live cohort** and **pass the Capstone project**.\n\nCertificates are **not awarded for self-paced mode**, because you need to **peer-review 3 capstones** after submitting your project, and that can only be done while the course is running. If you missed the first homework, that’s okay—homework is **not mandatory** for the certificate.'

In [69]:
assistant.rag("Can I still join the course after it started?")

'Yes, you can still join after the course has started. You can start whenever you want, but if you want a certificate, you need to finish the course with a live cohort and submit your project while submissions are still being accepted.'